# Financial reports: full-corpus named entities and actual retrieval

Process all 30 original reports / 26,058 source blocks with five downloadable open entity models. Use a free Colab T4 GPU. Predictions build actual per-model entity indexes and entity-assisted passage searches, compared with the original word-matching search. No private inference API is used.

Accuracy is scored only on the explicitly identified, independently reviewed reference sample; full-corpus processing coverage is reported separately. Agent-authored reference annotations are not a human financial gold standard. LangSmith records the small evaluation/retrieval series, not every full-corpus block, to limit hosted trace volume. Full report text and raw predictions remain in this private runtime and the local result download.


In [ ]:
# Runtime > Change runtime type > T4 GPU (free). No paid runtime is required.
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],check=True)
ROOT=Path('/content/financial-ner-workbench')
ROOT.mkdir(parents=True,exist_ok=True)


In [ ]:
# Upload the matching archive via the Files sidebar, or use the upload prompt below. Add a saved checkpoint when resuming.
from google.colab import files
EXPECTED_ARCHIVE_SHA256='d258d8963078e08f544f70aeecc2d8a0d787e366e90fcf157325fd2ea7c4c4e5'
preuploaded=Path('/content/workbench-ner-colab.zip')
uploaded={str(p):None for p in [preuploaded,*Path('/content').glob('workbench-ner-checkpoint-*.zip')]} if preuploaded.exists() else files.upload()
archive=next((Path(name) for name in uploaded if name.endswith('.zip') and hashlib.sha256(Path(name).read_bytes()).hexdigest()==EXPECTED_ARCHIVE_SHA256),None)
if archive is None: raise RuntimeError('Upload the matching workbench-ner-colab.zip; its fingerprint must match this notebook.')
def extract_checked(path):
    with zipfile.ZipFile(path) as z:
        for item in z.infolist():
            resolved=(ROOT/item.filename).resolve()
            if ROOT.resolve() not in resolved.parents: raise RuntimeError('Unexpected archive path')
        z.extractall(ROOT)
extract_checked(archive)
for name in uploaded:
    checkpoint=Path(name)
    if checkpoint!=archive and checkpoint.name.startswith('workbench-ner-checkpoint-') and checkpoint.suffix=='.zip': extract_checked(checkpoint)
os.chdir(ROOT)
corpus=json.loads((ROOT/'artifacts/local/corpus.json').read_text())
assert len(corpus['documents'])==30 and len(corpus['blocks'])==26058
print('Frozen input:',len(corpus['documents']),'reports /',len(corpus['blocks']),'source blocks')
del uploaded,corpus


In [ ]:
# Model-specific environments are isolated by the runner.
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface-hub>=0.34','langsmith>=0.14,<1','scikit-learn>=1.5'],check=True)


In [ ]:
# LangSmith is observability only. Model inference uses downloaded weights.
# Keep the key in Colab Secrets; never paste it into source or output.
USE_LANGSMITH=True
if USE_LANGSMITH:
    from google.colab import userdata
    os.environ['LANGSMITH_API_KEY']=userdata.get('LANGSMITH_API_KEY')
    os.environ['LANGSMITH_TRACING']='false' # full-corpus workers do not emit one hosted trace per block
    os.environ['LANGSMITH_PROJECT']='financial-research-workbench-ner'


In [ ]:
# Full corpus: no document/block limit. Model files/libraries are excluded from downloads.
OUT=ROOT/'artifacts/local/ner'
ARTIFACT_NAMES={'manifest.json','summary.json','metadata.json','status.json','predictions.jsonl','entity-index.json','retrieval.json','reference-predictions.jsonl','query-predictions.jsonl','evaluation.json','langsmith.json'}
def checkpoint_zip(name):
    path=Path('/content')/name
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUT.rglob('*'):
            if p.is_file() and p.name in ARTIFACT_NAMES and 'environments' not in p.parts and 'pinned-load' not in p.parts: z.write(p,p.relative_to(ROOT).as_posix())
        public=ROOT/'web/public/data/ner-results.json'
        if public.exists(): z.write(public,public.relative_to(ROOT).as_posix())
    return path
command=[sys.executable,'-u','scripts/ner_colab_run.py','--corpus','artifacts/local/corpus.json','--output','artifacts/local/ner','--models','all','--device','cuda','--batch-size','16','--reference','data/ner-reference-sample.json','--resume']
if USE_LANGSMITH: command+=['--langsmith']
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
finished=set()
for line in process.stdout:
    print(line,end='',flush=True)
    try: event=json.loads(line)
    except (ValueError,TypeError): continue
    if event.get('status')=='complete' and event.get('model_id') and event['model_id'] not in finished:
        finished.add(event['model_id'])
        files.download(str(checkpoint_zip('workbench-ner-checkpoint-'+str(len(finished)).zfill(2)+'.zip')))
exit_code=process.wait()
if exit_code: raise RuntimeError('One or more models did not complete. Inspect saved status and fix before publication.')


In [ ]:
# Download the final actual artifacts. No placeholder scores are filled in.
for p in sorted(OUT.glob('*/status.json')): print(p.parent.name,json.loads(p.read_text()))
files.download(str(checkpoint_zip('workbench-ner-results.zip')))
